# Feature Engineering and Analysis

### Import Libraries

In [ ]:
import fiona
import geopandas as gpd
import numpy as np
import os
import pandas as pd
from pathlib import Path

## 1. Load the Transshipment Datasets

In [ ]:
# CWD is the notebooks directory, as that is where Jupyter will run its kernel in.
TRANSSHIPMENT_DIR_FROM_CWD = "../data/raw/transshipment"
ENCOUNTER_EVENTS_FILENAME = "encounter-events-v20170717.csv"
LOITERING_EVENTS_FILENAME = "loitering-events-v20170717.csv"
VESSELS_FILENAME = "transshipment-vessels-v20170717.csv"

In [ ]:
transshipment_data_dir = os.path.join(os.getcwd(), TRANSSHIPMENT_DIR_FROM_CWD)

encounter_path = os.path.join(transshipment_data_dir, ENCOUNTER_EVENTS_FILENAME)
loitering_path = os.path.join(transshipment_data_dir, LOITERING_EVENTS_FILENAME)
transshipment_path = os.path.join(transshipment_data_dir, VESSELS_FILENAME)

encounter_df = pd.read_csv(encounter_path)
loitering_df = pd.read_csv(loitering_path)
transshipment_df = pd.read_csv(transshipment_path)


print("="*20 + " Encounter " + "="*20)
print("Shape: ",encounter_df.shape)
print("Example Row: ",encounter_df.head(1))
print("="*20 + "=========" + "="*20)

print()

print("="*20 + " Loitering " + "="*20)
print("Shape: ",loitering_df.shape)
print("Example Row: ",loitering_df.head(1))
print("="*20 + "=========" + "="*20)

print()

print("="*20 + " Transshipment " + "="*20)
print("Shape: ",transshipment_df.shape)
print("Example Row: ",transshipment_df.head(1))
print("="*20 + "=========" + "="*20)

## 2. Load the Global Marine Protected Area Data

In [ ]:
MPA_GDB_PATH_FROM_CWD = "../data/raw/global_mpa/WDPA_WDOECM_Apr2026_Public_marine.gdb"

In [ ]:
mpa_gdb_path = os.path.join(os.getcwd(), MPA_GDB_PATH_FROM_CWD)

fiona.listlayers(mpa_gdb_path)

In [ ]:
mpa_point_gdf = gpd.read_file(mpa_gdb_path, layer="WDPA_WDOECM_point_Apr2026_marine")

In [ ]:
mpa_location = mpa_point_gdf[['NAME_ENG', 'geometry']]

# Extract latitude and longitude
mpa_location['geometry'] = mpa_location['geometry'].explode(ignore_index=True)
mpa_location['lat'] = mpa_location['geometry'].y.astype(float)
mpa_location['long'] = mpa_location['geometry'].x.astype(float)
mpa_location.head()

## 3. Engineer Features
### Loitering Events

In [ ]:
loitering_new = loitering_df.copy()

#### 1. Add Distance Traveled

Distance traveled exposes how much distance a transshipment vessel loitered for. This could hint at the sort of activity that the vessel is engaging in, especially when paired with duration. For example, low distance traveled over a long duration suggests a transshipment vessel might be waiting to or has encountered a ship.

In [ ]:
loitering_new['diff_lat'] = loitering_new['ending_latitude'] - loitering_new['starting_latitude']
loitering_new['diff_long'] = loitering_new['ending_longitude'] - loitering_new['starting_longitude']
loitering_new['movement_mag'] = np.sqrt(
    loitering_new['diff_lat']**2 + loitering_new['diff_long']**2
)

#### 2. Add Whether Event was at Night

Loitering at nighttime, especially near marine protected areas, raises concern because vessels may be trying to hide encounters that involve illegal cargo.

In [ ]:
# Determine if activity occured during night by correcting time (Local solar hour = UTC + longitude/15) % 24
loitering_new['starting_timestamp'] = pd.to_datetime(loitering_new['starting_timestamp'])
loitering_new['local_hour'] = (
    loitering_new['starting_timestamp'].dt.hour +
    loitering_new['starting_longitude'] / 15
) % 24

# Redefine is_night (True if between 6 PM and 6 AM)
loitering_new['is_night_corrected'] = (
    (loitering_new['local_hour'] >= 18) |
    (loitering_new['local_hour'] < 6)
).astype(int)

#### 3. Add Distance to Nearest Marine Protected Area

This informs how close a transshipment vessel is loitering near a marine protected area. Small distances suggest a vessel could be waiting for ships carrying illegal cargo from the protected area, thus requring a closer look.

In [ ]:
loitering_new['nearest_mpa_name'] = ""
loitering_new['nearest_mpa_lat'] = 0
loitering_new['nearest_mpa_long'] = 0
loitering_new['nearest_mpa_dist'] = 0

loit_coords = loitering_new[['starting_latitude', 'starting_longitude']].values
mpa_coords = mpa_location[['lat', 'long']].values

indices = [np.linalg.norm(mpa_coords - point, axis=1).argmin() for point in loit_coords]

nearest_mpas = mpa_location.iloc[indices]
loitering_new['nearest_mpa_name'] = nearest_mpas['NAME_ENG'].values
loitering_new['nearest_mpa_lat'] = nearest_mpas['lat'].values
loitering_new['nearest_mpa_long'] = nearest_mpas['long'].values

loitering_new['nearest_mpa_dist'] = np.sqrt(
    (loitering_new['starting_latitude'] - loitering_new['nearest_mpa_lat'])**2 +
    (loitering_new['starting_longitude'] - loitering_new['nearest_mpa_long'])**2
)

In [ ]:
loitering_new.info()

## 4. Save Engineered Datasets

In [ ]:
PROCESSED_TRSHMNT_DIR_FROM_CWD = "../data/processed/transshipment"
PROCESSED_LOITERING_FILENAME = "processed-loitering-events.csv"

cwd_path = Path(os.getcwd())
processed_trshmnt_path = cwd_path / PROCESSED_TRSHMNT_DIR_FROM_CWD

processed_trshmnt_path.mkdir(parents=True, exist_ok=True)

In [ ]:
new_loitering_path = processed_trshmnt_path / PROCESSED_LOITERING_FILENAME
loitering_new.to_csv(new_loitering_path, index=False)

## 5. Visualize Vessel Proximity to Marine Protected Areas (MPA)

In [ ]:
import pandas as pd
import numpy as np
import folium
from dash import Dash, html, dcc, Input, Output
import plotly.express as px
import plotly.graph_objects as go

# 1. Initialize the Dash App
app = Dash(__name__)

loitering_nearest_mpa = loitering_new.copy()

# --- UI LAYOUT ---
app.layout = html.Div([
    html.H1("MPA Loitering & Transshipment Analytics", style={'textAlign': 'center', 'fontFamily': 'Arial'}),
    
    html.Div([
        html.Label("Select a Marine Protected Area (MPA):"),
        dcc.Dropdown(
            id='mpa-dropdown',
            options=[{'label': name, 'value': name} for name in loitering_nearest_mpa['nearest_mpa_name'].unique()],
            value=loitering_nearest_mpa['nearest_mpa_name'].iloc[0], # Default value
            clearable=False
        )
    ], style={'width': '50%', 'margin': 'auto', 'padding': '20px'}),

    # Statistics Row
    html.Div(id='stats-container', style={'display': 'flex', 'justifyContent': 'space-around', 'marginBottom': '20px'}),

    # Main Visuals Row
    html.Div([
        # Left: Folium Map
        html.Div([
            html.Iframe(id='folium-map', srcDoc="", width='100%', height='500')
        ], style={'width': '60%', 'display': 'inline-block'}),
    ])
])

# --- CALLBACKS ---
@app.callback(
    [Output('stats-container', 'children'),
     Output('folium-map', 'srcDoc')],
    [Input('mpa-dropdown', 'value')]
)
def update_dashboard(selected_mpa):
    # Filter data for the selected MPA
    filtered_df = loitering_nearest_mpa[loitering_nearest_mpa['nearest_mpa_name'] == selected_mpa]
    
    # 1. Calculate Statistics
    num_ships = filtered_df['transshipment_mmsi'].nunique()
    avg_duration = filtered_df['total_event_duration'].mean()
    total_events = len(filtered_df)
    
    stats_cards = [
        html.Div([html.H3("Unique Vessels"), html.P(num_ships)], className="card"),
        html.Div([html.H3("Avg Duration (Hrs)"), html.P(f"{avg_duration:.2f}")], className="card"),
        html.Div([html.H3("Total Events"), html.P(total_events)], className="card")
    ]

    # 2. Generate Folium Map
    # Center map on the MPA coordinates
    mpa_lat = filtered_df['nearest_mpa_lat'].iloc[0]
    mpa_lon = filtered_df['nearest_mpa_long'].iloc[0]
    
    m = folium.Map(location=[mpa_lat, mpa_lon], zoom_start=6, tiles='CartoDB positron')
    
    # Add MPA Marker
    folium.Marker(
        [mpa_lat, mpa_lon], 
        tooltip="MPA Center", 
        icon=folium.Icon(color='green', icon='leaf')
    ).add_to(m)

    # Add Ship Markers and Connection Lines
    for _, row in filtered_df.iterrows():
        # Ship location
        folium.CircleMarker(
            location=[row['starting_latitude'], row['starting_longitude']],
            radius=5, color='red', fill=True,
            popup=f"MMSI: {row['transshipment_mmsi']}<br>Duration: {row['total_event_duration']:.1f}h"
        ).add_to(m)
        
        # Line to MPA
        folium.PolyLine(
            locations=[[row['starting_latitude'], row['starting_longitude']], [mpa_lat, mpa_lon]],
            color="blue", weight=1, opacity=0.3
        ).add_to(m)
    
    map_html = m._repr_html_()

    return stats_cards, map_html

# 4. Run the app
if __name__ == '__main__':
    app.run(debug=True)